1. The Tensor class. 

The mathematical difference between the scaler Value is that gradients are no longer numbers, they're arrays of same shape as the tensor. 
When tensor of different shapes interact, gradients must be summed back down to original shape. 

For C = A + B where B is broadcast (eg bias vector added to a batch). 
A shape = (N, D)
B shape = (D, )

$$\frac{\partial L}{\partial B_j} = \sum_{i=1}^{N} \frac{\partial L}{\partial C_{ij}}$$

i.e I need to sum the incoming gradient over the broadcasted axes. This is only the new idea here, the chain rule itself is identical to what I already implemented in micrograd. 

In [1]:
import numpy as np
class Tensor:
    def __init__(self, data, _children=(), _op=''):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
    
    @property
    def shape(self):
        return self.data.shape
    
    def __repr__(self):
        return f"Tensor(data={self.data})"

    def _unbroadcast(self, grad, shape):
        # grad shape (3, ), target shape (). 
        # sums across the rows
        
        while grad.ndim > len(shape):
            grad = grad.sum(axis=0)
        
        # checks for 1 dimensions in target shape and sum across that axis.
        # if target shape is (1, 3) sums over that axis and keeps the dimension as (1, 3) 
        for i, dim in enumerate(shape):
            if dim == 1:
                grad = grad.sum(axis=i, keepdims=True)
        return grad

    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += self._unbroadcast(out.grad, self.data.shape)
            other.grad += self._unbroadcast(out.grad, other.data.shape)
        out._backward = _backward
        
        return out
    
    def __radd__(self, other):
        return self + other
        



In [2]:
a = Tensor([1.0, 2.0, 3.0])
b = Tensor(2.0)
c = b + a
c

Tensor(data=[3. 4. 5.])

In [3]:
print(f"a shape: {a.shape}, b shape: {b.shape}")

a shape: (3,), b shape: ()


In [4]:
c.grad = np.array([1.0, 1.0, 1.0])
c._backward()
print(a.grad, b.grad)

[1. 1. 1.] 3.0


Since b was added to all 3 elements of a, it's gradient is the sum of all incoming gradients (1.0 + 1.0 + 1.0) = 3.0. 


In [5]:
d = Tensor([
    [1.0, 2.0],
    [3.0, 4.0]
])

e = Tensor([10.0, 20.0])
f = d + e
print("d shape", d.shape)
print("e shape", e.shape)
print("f shape", f.shape)
f

d shape (2, 2)
e shape (2,)
f shape (2, 2)


Tensor(data=[[11. 22.]
 [13. 24.]])

In [6]:
f.grad = np.array([
    [1.0, 2.0],
    [3.0, 4.0]
])

f._backward()

print("d grad: ", d.grad)
print("e grad: ", e.grad)

d grad:  [[1. 2.]
 [3. 4.]]
e grad:  [4. 6.]


here, 10.0 was added to both 1.0 and 3.0 so it must have gradient of both 1.0 and 3.0. (1.0 + 3.0 = 4.0)



In [7]:
g = Tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

h = Tensor([
    [10.0, 20.0, 30.0]
])

i = g + h
print("g shape: ", g.shape)
print("h shape: ", h.shape)
print("i shape: ", i.shape)

g shape:  (2, 3)
h shape:  (1, 3)
i shape:  (2, 3)


numpy did this while adding. 

$$i = \begin{bmatrix} 1.0 & 2.0 & 3.0 \\ 4.0 & 5.0 & 6.0 \end{bmatrix} + \begin{bmatrix} 10.0 & 20.0 & 30.0 \\ 10.0 & 20.0 & 30.0 \end{bmatrix} = \begin{bmatrix} 11.0 & 22.0 & 33.0 \\ 14.0 & 25.0 & 36.0 \end{bmatrix}$$

In [8]:
# suppose the incoming grad for i is. 
i.grad = np.array([
    [1.0, 1.0, 1.0],
    [2.0, 2.0, 2.0],
])

i._backward()

print("g grad: ", g.grad)
print("i grad: ", h.grad)

g grad:  [[1. 1. 1.]
 [2. 2. 2.]]
i grad:  [[3. 3. 3.]]


In [9]:
# a complex multi-dimensional tensor shape
A = Tensor(np.ones((2, 4, 3)))

B_data = np.array([
    [[10.0, 20.0, 30.0]],
    [[40.0, 50.0, 60.0]]
])

B = Tensor(B_data)

C = A + B

print("A shape: ", A.shape)
print("B shape: ", B.shape)
print("C shape: ", C.shape)

A shape:  (2, 4, 3)
B shape:  (2, 1, 3)
C shape:  (2, 4, 3)


In [10]:
A

Tensor(data=[[[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]])

In [12]:
C

Tensor(data=[[[11. 21. 31.]
  [11. 21. 31.]
  [11. 21. 31.]
  [11. 21. 31.]]

 [[41. 51. 61.]
  [41. 51. 61.]
  [41. 51. 61.]
  [41. 51. 61.]]])

In [13]:
# suppose upstream gradient is. 
C.grad = np.ones((2, 4, 3))
C._backward()

print("A grad: ", A.grad)
print("B grad: ", B.grad)

A grad:  [[[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]

 [[1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]
  [1. 1. 1.]]]
B grad:  [[[4. 4. 4.]]

 [[4. 4. 4.]]]
